# Training on Colab

Runs `src/train.py` on a Colab GPU and writes results to Google Drive. The published
runs used fp16 on a Tesla T4 and an A100.

While the repository is private, add a GitHub personal access token with read access
to it as a Colab secret named `GITHUB_TOKEN` (key icon in the left sidebar). After the
install cell, restart the runtime once and continue from the next cell. Run the
training commands through these cells; do not paste `train.py` into a cell, because it
imports `src/data.py` and only finds it when run as a script.


In [ ]:
import os, subprocess
REPO = "github.com/ArmanMahmud003/bangla-nli-lora"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
if not os.path.isdir("/content/repo"):
    r = subprocess.run(["git", "clone", "-q", url, "/content/repo"],
                       capture_output=True, text=True)
    if r.returncode:
        raise SystemExit("clone failed (private repo? set the GITHUB_TOKEN secret)")
!git -C /content/repo pull -q
!pip install -q -r /content/repo/requirements.txt
!pip install -q git+https://github.com/csebuetnlp/normalizer


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/bangla_nli_runs"
!nvidia-smi --query-gpu=name,memory.total --format=csv

Pilot: one seed at 1% for both methods (about 10 minutes on an A100). Test macro-F1
should land around 0.71 to 0.73.

In [ ]:
!cd /content/repo && python src/train.py --out {OUT} --scratch /content/ckpt --fractions 0.01 --seeds 42

Full grid. Finished runs are skipped, so the cell can be re-run after a disconnect.

In [ ]:
!cd /content/repo && python src/train.py --out {OUT} --scratch /content/ckpt

In [ ]:
!cd /content/repo && python src/analyze.py --artifacts {OUT} --figures {OUT}/figures